# **Deep Residual MLP**

## **1. Import Libraries**

In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns


## **2. Data Preparation**

In [7]:
import os

def data_preparation(file_path, seed=42):
    df = pd.read_csv(file_path)
    df = df.drop(columns=["CustomerID", "AdvertisingPlatform", "AdvertisingTool"], errors="ignore")

    le = LabelEncoder()
    df["CampaignType"] = le.fit_transform(df["CampaignType"])

    features = [
        "CampaignType", "AdSpend", "ClickThroughRate", "WebsiteVisits", "PagesPerVisit",
        "TimeOnSite", "EmailOpens", "EmailClicks", "PreviousPurchases", "LoyaltyPoints"
    ]
    X = df[features]
    y = df["Conversion"]

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=seed, stratify=y_temp)
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = data_preparation('../dataset/digital_marketing_campaign_dataset.csv')

## **3. Preprocessing**

In [8]:
features = [
    'AdSpend', 'ClickThroughRate', 'WebsiteVisits', 'PagesPerVisit',
    'TimeOnSite', 'EmailOpens', 'EmailClicks', 'PreviousPurchases', 'LoyaltyPoints', 'CampaignType'
]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train[features])
X_val   = scaler.transform(X_val[features])
X_test  = scaler.transform(X_test[features])


## **4. Hyperparameter Configuration**

In [9]:
HIDDEN_DIM  = 16
NUM_BLOCKS  = 1
HEAD_DIM    = 8
DROPOUT     = 0.7
BATCH_SIZE  = 128
LR          = 3e-4
EPOCHS      = 120
PATIENCE    = 15
LR_FACTOR   = 0.5
LR_PATIENCE = 5
LR_MIN      = 1e-5


## **5. Training**

In [10]:
def residual_block(x, units, dropout):
    h = Dense(units, activation='relu')(x)
    h = BatchNormalization()(h)
    h = Dropout(dropout)(h)
    h = Dense(units)(h)
    h = BatchNormalization()(h)
    return tf.keras.layers.Activation('relu')(Add()([x, h]))


inputs = Input(shape=(X_train.shape[1],))
x = Dense(HIDDEN_DIM, activation='relu')(inputs)
x = BatchNormalization()(x)
for _ in range(NUM_BLOCKS):
    x = residual_block(x, HIDDEN_DIM, DROPOUT)
x = Dense(HEAD_DIM, activation='relu')(x)
x = Dropout(DROPOUT)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=PATIENCE, restore_best_weights=True, mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN, mode='max'),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/120
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.6336 - auc: 0.4913 - loss: 1.1026 - val_accuracy: 0.7000 - val_auc: 0.4801 - val_loss: 0.6015 - learning_rate: 3.0000e-04
Epoch 2/120
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6748 - auc: 0.4989 - loss: 0.9484 - val_accuracy: 0.7406 - val_auc: 0.4931 - val_loss: 0.5669 - learning_rate: 3.0000e-04
Epoch 3/120
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6893 - auc: 0.5006 - loss: 0.8873 - val_accuracy: 0.7664 - val_auc: 0.5041 - val_loss: 0.5430 - learning_rate: 3.0000e-04
Epoch 4/120
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7184 - auc: 0.4978 - loss: 0.8191 - val_accuracy: 0.7906 - val_auc: 0.5182 - val_loss: 0.5249 - learning_rate: 3.0000e-04
Epoch 5/120
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7326 - auc: 0.5287 - loss: 0.7561 - val_accuracy: 0.8039 - val_auc: 0.5326 - val_loss: 0.5095 - learning_rate: 3.0000e-04
Epoch 6/120
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0

## **6. Predict Testing Dataset**

In [11]:
y_train_prob = model.predict(X_train).flatten()
y_test_prob  = model.predict(X_test).flatten()


160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


## **7. Evaluation**

In [12]:
y_pred        = (y_test_prob >= 0.5).astype(int)
acc           = accuracy_score(y_test, y_pred)
roc_auc_train = roc_auc_score(y_train, y_train_prob)
roc_auc_test  = roc_auc_score(y_test,  y_test_prob)
auc_gap       = roc_auc_train - roc_auc_test

summary = pd.DataFrame({
    'Metric': ['Accuracy (Test)', 'ROC-AUC (Train)', 'ROC-AUC (Test)', 'AUC Gap (Train-Test)'],
    'Score':  [round(acc, 4), round(roc_auc_train, 4), round(roc_auc_test, 4), round(auc_gap, 4)],
})
summary


,Metric,Score
0,Accuracy (Test),0.8762
1,ROC-AUC (Train),0.7817
2,ROC-AUC (Test),0.7380
3,AUC Gap (Train-Test),0.0437


In [13]:
report_df = pd.DataFrame(
    classification_report(y_test, y_pred, target_names=['Not Converted (0)', 'Converted (1)'], output_dict=True)
).T.round(4)
report_df


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

,precision,recall,f1-score,support
Not Converted (0),0.0000,0.0000,0.0000,198.0000
Converted (1),0.8762,1.0000,0.9340,1402.0000
accuracy,0.8762,0.8762,0.8762,0.8762
macro avg,0.4381,0.5000,0.4670,1600.0000
weighted avg,0.7678,0.8762,0.8185,1600.0000
